# 🎯 Clase 1: Prompt Engineering & Vibe Check

## Bienvenido al Taller de IA - Semana 1, Clase 1

En esta clase aprenderás:
- ✅ Cómo funcionan los Large Language Models (LLMs)
- ✅ Conceptos fundamentales: temperatura, top-P, top-K, tokens
- ✅ Tokenización y context window
- ✅ Técnicas de prompt engineering
- ✅ Mejores prácticas para obtener mejores resultados

---

## 📦 Instalación y Configuración

Primero, instalemos las librerías necesarias:

In [ ]:
# Instalación de dependencias
!pip install openai anthropic python-dotenv tiktoken -q

In [ ]:
# Imports necesarios
import os
from dotenv import load_dotenv
import tiktoken
from openai import OpenAI
import json

# Cargar variables de entorno
load_dotenv()

# Inicializar cliente de OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Configuración completada")

---

## 🧠 Parte 1: ¿Qué es un LLM?

### Conceptos Clave

Un **Large Language Model (LLM)** es un modelo de inteligencia artificial entrenado con enormes cantidades de texto para:
- Predecir la siguiente palabra/token más probable
- Generar texto coherente y contextual
- Responder preguntas, resumir, traducir, y mucho más

### ¿Cómo funciona?

1. **Input (Entrada)**: Recibes un prompt (texto)
2. **Tokenización**: El texto se convierte en tokens (números)
3. **Procesamiento**: El modelo procesa los tokens usando redes neuronales
4. **Predicción**: Genera tokens de salida uno por uno
5. **Output (Salida)**: Los tokens se convierten de vuelta a texto

```
Texto → Tokens → Modelo → Tokens → Texto
```

---

## 🔢 Parte 2: Tokens y Tokenización

### ¿Qué es un Token?

Un **token** es la unidad básica que los LLMs procesan. Puede ser:
- Una palabra completa: `"hola"` = 1 token
- Parte de una palabra: `"increíble"` = 2 tokens (`"incre"`, `"íble"`)
- Un carácter especial: `"!"` = 1 token

**Regla general**: 1 token ≈ 4 caracteres en español (0.75 palabras aprox.)

In [ ]:
# Función para contar tokens
def contar_tokens(texto, modelo="gpt-4"):
    """Cuenta cuántos tokens tiene un texto."""
    encoding = tiktoken.encoding_for_model(modelo)
    tokens = encoding.encode(texto)
    return len(tokens), tokens

# Ejemplos
textos_ejemplo = [
    "Hola",
    "Hola, ¿cómo estás?",
    "La inteligencia artificial está revolucionando el mundo.",
    "¡Increíble! 🚀"
]

print("📊 Análisis de Tokenización:\n")
for texto in textos_ejemplo:
    num_tokens, tokens = contar_tokens(texto)
    print(f"Texto: '{texto}'")
    print(f"  → Tokens: {num_tokens}")
    print(f"  → Caracteres: {len(texto)}")
    print(f"  → Ratio: {len(texto)/num_tokens:.2f} caracteres/token\n")

### 💡 Ejercicio 1: Experimenta con Tokenización

Prueba con tus propios textos:

In [ ]:
# 👉 Personaliza este texto
mi_texto = "Escribe aquí tu propio texto para analizar"

num_tokens, tokens = contar_tokens(mi_texto)
print(f"Tu texto tiene {num_tokens} tokens")
print(f"Estimación de costo (GPT-4): ${num_tokens * 0.00003:.6f} USD")

---

## 🌡️ Parte 3: Temperatura y Parámetros de Generación

### Temperatura (Temperature)

Controla la **aleatoriedad** de las respuestas:

- **Temperatura = 0**: Determinista, siempre la misma respuesta
  - Uso: Tareas que requieren precisión (matemáticas, código, extracción de datos)
  
- **Temperatura = 0.7**: Balance entre creatividad y coherencia
  - Uso: Conversaciones generales, asistentes
  
- **Temperatura = 1.5-2**: Muy creativo, impredecible
  - Uso: Generación creativa, brainstorming, escritura artística

### Top-P (Nucleus Sampling)

Controla la **diversidad** considerando solo los tokens más probables:
- **top_p = 0.1**: Solo los tokens más probables (conservador)
- **top_p = 0.9**: Mayor diversidad de opciones

### Top-K

Limita la selección a los **K tokens más probables**:
- **top_k = 10**: Solo considera las 10 opciones más probables
- **top_k = 50**: Mayor variedad

In [ ]:
# Función helper para llamar al LLM
def llamar_llm(prompt, temperatura=0.7, max_tokens=150):
    """Llama al LLM con los parámetros especificados."""
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperatura,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

# Experimentar con diferentes temperaturas
prompt_prueba = "Escribe una historia corta sobre un robot que aprende a cocinar."

temperaturas = [0, 0.7, 1.5]

print("🌡️ Comparación de Temperaturas:\n")
print("="*80)

for temp in temperaturas:
    print(f"\n📊 TEMPERATURA = {temp}")
    print("-"*80)
    respuesta = llamar_llm(prompt_prueba, temperatura=temp)
    print(respuesta)
    print("="*80)

### 💡 Ejercicio 2: Experimenta con Temperatura

Prueba el mismo prompt con diferentes temperaturas:

In [ ]:
# 👉 Personaliza este prompt
mi_prompt = "Dame 3 ideas innovadoras para una startup de IA"

# 👉 Prueba con diferentes temperaturas
mi_temperatura = 0.7  # Cambia este valor entre 0 y 2

respuesta = llamar_llm(mi_prompt, temperatura=mi_temperatura)
print(f"Temperatura: {mi_temperatura}\n")
print(respuesta)

---

## 🪟 Parte 4: Context Window (Ventana de Contexto)

El **context window** es la cantidad máxima de tokens que el modelo puede "recordar" en una conversación.

### Límites por Modelo:

| Modelo | Context Window | Equivalente Aprox. |
|--------|----------------|--------------------|
| GPT-3.5 | 4,096 tokens | ~3,000 palabras |
| GPT-4 | 8,192 tokens | ~6,000 palabras |
| GPT-4-32k | 32,768 tokens | ~24,000 palabras |
| Claude 3 | 200,000 tokens | ~150,000 palabras |

### ¿Qué pasa si excedes el límite?

- ❌ El modelo "olvida" la información más antigua
- ❌ Puede generar errores o respuestas incompletas
- ✅ Solución: Usar técnicas como RAG (¡lo veremos en Clase 2!)

In [ ]:
# Verificar el tamaño de una conversación
conversacion = [
    {"role": "system", "content": "Eres un asistente útil y amigable."},
    {"role": "user", "content": "¿Qué es la inteligencia artificial?"},
    {"role": "assistant", "content": "La inteligencia artificial es..."},
    {"role": "user", "content": "Dame ejemplos de aplicaciones"}
]

# Contar tokens de toda la conversación
total_tokens = 0
for mensaje in conversacion:
    num_tokens, _ = contar_tokens(mensaje["content"])
    total_tokens += num_tokens
    print(f"{mensaje['role']}: {num_tokens} tokens")

print(f"\n📊 Total de tokens en conversación: {total_tokens}")
print(f"📊 Porcentaje usado (GPT-4): {(total_tokens/8192)*100:.2f}%")

---

## ✍️ Parte 5: Prompt Engineering - Técnicas Fundamentales

### 1. Sé Claro y Específico

❌ **Mal**: "Háblame de IA"

✅ **Bien**: "Explica en 3 párrafos qué es la inteligencia artificial, enfocándote en aplicaciones prácticas en el sector salud"

In [ ]:
# Comparación: Prompt vago vs específico
prompt_vago = "Háblame de IA"
prompt_especifico = "Explica en 3 párrafos qué es la inteligencia artificial, enfocándote en aplicaciones prácticas en el sector salud"

print("❌ Prompt Vago:")
print(llamar_llm(prompt_vago, temperatura=0.7))
print("\n" + "="*80 + "\n")
print("✅ Prompt Específico:")
print(llamar_llm(prompt_especifico, temperatura=0.7))

### 2. Proporciona Contexto y Rol

Darle un "rol" al modelo mejora la calidad de las respuestas:

In [ ]:
# Sin rol
prompt_sin_rol = "¿Cómo puedo mejorar mi código Python?"

# Con rol
prompt_con_rol = """Eres un ingeniero de software senior experto en Python con 10 años de experiencia.
Un desarrollador junior te pregunta: ¿Cómo puedo mejorar mi código Python?
Proporciona 5 consejos prácticos y concretos."""

print("Sin rol:")
print(llamar_llm(prompt_sin_rol, temperatura=0.5))
print("\n" + "="*80 + "\n")
print("Con rol:")
print(llamar_llm(prompt_con_rol, temperatura=0.5))

### 3. Few-Shot Learning (Aprendizaje con Ejemplos)

Proporciona ejemplos de lo que quieres:

In [ ]:
# Few-shot prompt para clasificación de sentimiento
prompt_few_shot = """Clasifica el sentimiento de las siguientes reseñas como Positivo, Negativo o Neutral.

Ejemplos:
Reseña: "¡Me encantó este producto! Superó mis expectativas."
Sentimiento: Positivo

Reseña: "No funcionó como esperaba, muy decepcionante."
Sentimiento: Negativo

Reseña: "Es un producto normal, nada especial."
Sentimiento: Neutral

Ahora clasifica esta:
Reseña: "La calidad es buena pero el precio es muy alto."
Sentimiento:"""

print(llamar_llm(prompt_few_shot, temperatura=0))

### 4. Chain of Thought (Cadena de Pensamiento)

Pide al modelo que "piense paso a paso":

In [ ]:
# Sin Chain of Thought
prompt_directo = "Si un tren viaja a 80 km/h y debe recorrer 240 km, ¿cuánto tiempo tardará?"

# Con Chain of Thought
prompt_cot = """Si un tren viaja a 80 km/h y debe recorrer 240 km, ¿cuánto tiempo tardará?
Piensa paso a paso y muestra tu razonamiento."""

print("Respuesta directa:")
print(llamar_llm(prompt_directo, temperatura=0))
print("\n" + "="*80 + "\n")
print("Con Chain of Thought:")
print(llamar_llm(prompt_cot, temperatura=0))

### 5. Formato de Salida Estructurado

Especifica el formato que deseas (JSON, tabla, lista, etc.):

In [ ]:
prompt_json = """Dame información sobre 3 lenguajes de programación populares.
Responde en formato JSON con esta estructura:
{
  "lenguajes": [
    {
      "nombre": "nombre del lenguaje",
      "año_creacion": año,
      "uso_principal": "descripción",
      "popularidad": "alta/media/baja"
    }
  ]
}"""

respuesta_json = llamar_llm(prompt_json, temperatura=0)
print(respuesta_json)

# Intentar parsear el JSON
try:
    datos = json.loads(respuesta_json)
    print("\n✅ JSON válido parseado correctamente")
    print(f"Se obtuvieron {len(datos['lenguajes'])} lenguajes")
except:
    print("\n⚠️ La respuesta no es JSON válido")

---

## 🎯 Parte 6: Mejores Prácticas de Prompt Engineering

### Checklist de un Buen Prompt:

✅ **Claro y específico**: Define exactamente qué quieres

✅ **Contexto adecuado**: Proporciona información relevante

✅ **Rol definido**: Dale una "personalidad" al modelo

✅ **Ejemplos (si es necesario)**: Muestra lo que esperas

✅ **Formato de salida**: Especifica cómo quieres la respuesta

✅ **Temperatura apropiada**: Ajusta según la tarea

✅ **Limitaciones claras**: Define restricciones (longitud, estilo, etc.)

### Template de Prompt Completo:

In [ ]:
# Template de prompt profesional
def crear_prompt_profesional(rol, tarea, contexto="", ejemplos="", formato="", restricciones=""):
    """Crea un prompt estructurado siguiendo mejores prácticas."""
    prompt = f"""ROL: {rol}

TAREA: {tarea}
"""
    
    if contexto:
        prompt += f"\nCONTEXTO: {contexto}\n"
    
    if ejemplos:
        prompt += f"\nEJEMPLOS:\n{ejemplos}\n"
    
    if formato:
        prompt += f"\nFORMATO DE SALIDA: {formato}\n"
    
    if restricciones:
        prompt += f"\nRESTRICCIONES: {restricciones}\n"
    
    return prompt

# Ejemplo de uso
prompt_profesional = crear_prompt_profesional(
    rol="Eres un experto en marketing digital con especialización en redes sociales.",
    tarea="Crea 3 posts para LinkedIn sobre inteligencia artificial en empresas.",
    contexto="La audiencia son CEOs y directores de tecnología de empresas medianas.",
    formato="Cada post debe tener: título llamativo, cuerpo de 2-3 párrafos, y 3 hashtags relevantes.",
    restricciones="Máximo 200 palabras por post. Tono profesional pero accesible."
)

print("📝 Prompt Generado:")
print("="*80)
print(prompt_profesional)
print("="*80)
print("\n🤖 Respuesta del LLM:")
print("="*80)
print(llamar_llm(prompt_profesional, temperatura=0.8))

---

## 💪 Ejercicios Prácticos

### Ejercicio 3: Crea tu Propio Asistente Especializado

In [ ]:
# 👉 Personaliza estos valores para crear tu asistente

mi_rol = "Eres un..."  # Define el rol de tu asistente
mi_tarea = ""  # ¿Qué quieres que haga?
mi_contexto = ""  # Información adicional relevante
mi_formato = ""  # ¿Cómo quieres la respuesta?

mi_prompt = crear_prompt_profesional(
    rol=mi_rol,
    tarea=mi_tarea,
    contexto=mi_contexto,
    formato=mi_formato
)

print(llamar_llm(mi_prompt, temperatura=0.7))

### Ejercicio 4: Optimización de Prompts

Toma un prompt básico y mejóralo aplicando las técnicas aprendidas:

In [ ]:
# Prompt básico (mejóralo)
prompt_basico = "Dame ideas para un negocio"

# 👉 Tu prompt mejorado aquí
prompt_mejorado = """
# Escribe aquí tu versión mejorada del prompt
"""

print("Prompt Básico:")
print(llamar_llm(prompt_basico, temperatura=0.7))
print("\n" + "="*80 + "\n")
print("Prompt Mejorado:")
print(llamar_llm(prompt_mejorado, temperatura=0.7))

---

## 📊 Parte 7: Análisis de Costos

Entender los costos es crucial para usar LLMs en producción:

In [ ]:
# Precios aproximados (verificar precios actuales)
PRECIOS = {
    "gpt-4": {"input": 0.03, "output": 0.06},  # por 1K tokens
    "gpt-3.5-turbo": {"input": 0.0015, "output": 0.002},
    "claude-3-opus": {"input": 0.015, "output": 0.075}
}

def calcular_costo(prompt, respuesta, modelo="gpt-4"):
    """Calcula el costo aproximado de una llamada al LLM."""
    tokens_input, _ = contar_tokens(prompt, modelo)
    tokens_output, _ = contar_tokens(respuesta, modelo)
    
    costo_input = (tokens_input / 1000) * PRECIOS[modelo]["input"]
    costo_output = (tokens_output / 1000) * PRECIOS[modelo]["output"]
    costo_total = costo_input + costo_output
    
    return {
        "tokens_input": tokens_input,
        "tokens_output": tokens_output,
        "tokens_total": tokens_input + tokens_output,
        "costo_usd": costo_total
    }

# Ejemplo
prompt_ejemplo = "Explica qué es machine learning en 100 palabras"
respuesta_ejemplo = llamar_llm(prompt_ejemplo, temperatura=0.7)

costos = calcular_costo(prompt_ejemplo, respuesta_ejemplo)

print("💰 Análisis de Costos:")
print(f"Tokens de entrada: {costos['tokens_input']}")
print(f"Tokens de salida: {costos['tokens_output']}")
print(f"Total tokens: {costos['tokens_total']}")
print(f"Costo: ${costos['costo_usd']:.6f} USD")
print(f"\nCosto por 1,000 llamadas similares: ${costos['costo_usd'] * 1000:.2f} USD")

---

## 🎓 Resumen de la Clase

### Conceptos Aprendidos:

1. ✅ **LLMs**: Modelos que predicen tokens basándose en patrones aprendidos
2. ✅ **Tokens**: Unidad básica de procesamiento (~4 caracteres en español)
3. ✅ **Temperatura**: Controla creatividad (0 = determinista, 2 = muy creativo)
4. ✅ **Top-P y Top-K**: Controlan diversidad en la generación
5. ✅ **Context Window**: Límite de memoria del modelo
6. ✅ **Prompt Engineering**: Técnicas para obtener mejores resultados

### Técnicas de Prompt Engineering:

- 🎯 Claridad y especificidad
- 🎭 Asignación de roles
- 📚 Few-shot learning
- 🧠 Chain of thought
- 📋 Formato estructurado

### Próximos Pasos:

En la **Clase 2** aprenderemos:
- 🔍 Embeddings y búsqueda semántica
- 📚 RAG (Retrieval Augmented Generation)
- 🦙 Instalación de LLAMA local
- 💾 Bases de datos vectoriales

---

## 📚 Recursos Adicionales

- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic Prompt Library](https://docs.anthropic.com/claude/prompt-library)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [Tokenizer Playground](https://platform.openai.com/tokenizer)

---

## 🎯 Tarea para Casa

1. Experimenta con diferentes temperaturas en tus propios prompts
2. Crea 5 prompts profesionales usando el template aprendido
3. Analiza el costo de tus prompts más comunes
4. Lee el diccionario de términos de IA (`diccionario_ia.md`)

**¡Nos vemos en la Clase 2! 🚀**